# Weee-Fetch: 商品评价抓取工具 (Colab 版) 🛍️

**Weee-Fetch** 是一款强大的 Python 自动化工具，专门用于抓取 Weee! 平台的商品评价。它可以自动提取商品信息、处理中英双语、并将所有评价（包含高清大图）完美排版到 Excel 表格中。

---

### ✨ 核心功能
- **智能链接解析**：直接粘贴 Weee! 商品链接，程序会自动提取商品名和 ID 用于命名。
- **多语言支持**：支持抓取 **中文 (CN)** 或 **英文 (EN)** 的原始评价。
- **高清 Excel 报告**：自动将评价图片嵌入到表格单元格中，并优化行高列宽，方便直接查看。
- **自动打包下载**：所有数据处理完成后，会自动打包成 ZIP 文件并触发浏览器下载到本地。

### 📖 使用指南
1.  **第一步：环境准备** - 点击下方第一个代码单元格的播放按钮，安装必要的运行库。
2.  **第二步：参数配置** - 在下方的表单中粘贴商品链接，并选择您想要的语言。
3.  **第三步：开始运行** - 点击运行主代码块。等待进度条走完。
4.  **第四步：查看结果** - 程序结束后，浏览器会自动弹出 ZIP 压缩包下载。里面包含 JSON 和 Excel 文件。

> [!IMPORTANT]
> **免责声明：** 本项目仅供学习和研究使用。请尊重平台的服务条款，切勿用于任何商业用途或大规模恶意抓取。

---

## 🛠️ 第一步：环境配置
首次运行前请先执行此单元格以准备运行环境。

In [1]:
# @title 安装依赖库
!pip install requests openpyxl pillow tqdm

## 🔍 第二步：配置参数并开始抓取
在下方表单中填入信息后运行该单元格即可。
注意：如果想要抓取全部评论，请不要填写max_reviews_to_fetch 

**注意：** 抓取完成后，ZIP 文件会自动下载到您的电脑。

In [2]:
# @title 开始抓取 { display-mode: "form" }
product_url_or_id = "https://www.sayweee.com/zh/products/HADAY-Delicious-Light-Soy-Sauce/113281" # @param {type:"string"}
language = "cn" # @param ["cn", "en"]
max_reviews_to_fetch = 50 # @param {type:"integer"}
lite_mode = True # @param {type:"boolean"}

import requests
import json
import time
import io
import os
import re
from datetime import datetime
from openpyxl import Workbook
from openpyxl.drawing.image import Image as OpenpyxlImage
from openpyxl.utils import get_column_letter
from PIL import Image as PILImage
from tqdm.notebook import tqdm
from google.colab import files

def get_headers(product_id, lang="zh"):
    user_agent = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    if lang == "zh":
        accept_language = "zh-CN,zh;q=0.9,en;q=0.8"
        referer = f"https://www.sayweee.com/zh/product/view/{product_id}"
    else:
        accept_language = "en-US,en;q=0.9,zh;q=0.8"
        referer = f"https://www.sayweee.com/en/product/view/{product_id}"
    return {
        "User-Agent": user_agent,
        "Accept-Language": accept_language,
        "lang": lang,
        "Referer": referer,
    }

def fetch_all_reviews(product_id, max_reviews=None, lang='zh'):
    base_url = "https://api.sayweee.net/ec/social/review"
    all_reviews = []
    page = 1
    limit = 20
    
    pbar = tqdm(desc="正在抓取评论", unit="条")
    while True:
        params = {"product_id": product_id, "sort": "relevance", "page": page, "limit": limit}
        try:
            response = requests.get(base_url, params=params, headers=get_headers(product_id, lang))
            data = response.json()
            object_data = data.get('object') or {}
            reviews = object_data.get('list', [])
            
            if page == 1:
                total_from_api = object_data.get('total')
                if total_from_api:
                    pbar.total = min(total_from_api, max_reviews) if max_reviews else total_from_api
                    pbar.refresh()
            
            if not reviews: break
            
            added_reviews = min(len(reviews), max_reviews - len(all_reviews) if max_reviews else len(reviews))
            all_reviews.extend(reviews[:added_reviews])
            pbar.update(added_reviews)
            
            if max_reviews is not None and len(all_reviews) >= max_reviews: break
            page += 1
            time.sleep(0.5)
        except Exception as e:
            print(f"抓取第 {page} 页时出错: {e}")
            break
    pbar.close()
    return all_reviews

print("🚀 任务开始...")

# 1. 解析输入
product_name = ""
if product_url_or_id.startswith('http'):
    parts = product_url_or_id.split('?')[0].strip('/').split('/')
    product_id = parts[-1]
    product_name = parts[-2] if len(parts) >= 2 else ""
else:
    product_id = product_url_or_id

target_lang = "zh" if language == "cn" else "en"
if product_name:
    product_name = re.sub(r'[\\/:*?"<>| ]', '-', product_name)

# 2. 获取数据
reviews_data = fetch_all_reviews(product_id, max_reviews=max_reviews_to_fetch, lang=target_lang)

# 3. 保存文件夹
current_time = time.strftime("%Y%m%d_%H%M%S")
dir_name = f"{product_name}_{product_id}_{current_time}_{len(reviews_data)}" if product_name else f"{product_id}_{current_time}_{len(reviews_data)}"
if lite_mode: dir_name += "_lite"
os.makedirs(dir_name, exist_ok=True)

json_path = os.path.join(dir_name, f"{dir_name}.json")
excel_path = os.path.join(dir_name, f"{dir_name}.xlsx")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(reviews_data, f, ensure_ascii=False, indent=4)

if reviews_data:
    print("\n🎨 正在生成包含图片的 Excel 表格...")
    wb = Workbook()
    ws = wb.active
    ws.title = "评价详情"
    
    from datetime import datetime
    for review in reviews_data:
        if 'rec_create_time' in review and isinstance(review['rec_create_time'], int):
            try:
                review['rec_create_time'] = datetime.fromtimestamp(review['rec_create_time']).strftime('%Y-%m-%d %H:%M:%S')
            except: pass
    
    fieldnames = []
    for review in reviews_data:
        for key in review:
            if key not in fieldnames: fieldnames.append(key)

    if lite_mode:
        allowed = ['id', 'rec_create_time', 'comment', 'pictures']
        fieldnames = [k for k in fieldnames if k in allowed]

    max_pics = 0
    for review in reviews_data:
        pics = review.get('pictures')
        if isinstance(pics, list) and len(pics) > max_pics: max_pics = len(pics)

    if 'pictures' in fieldnames: fieldnames.remove('pictures')
    headers = fieldnames + [f"图片_{i+1}" for i in range(max_pics)]
    ws.append(headers)

    for i in range(max_pics):
        ws.column_dimensions[get_column_letter(len(fieldnames) + 1 + i)].width = 36

    for row_idx, review in enumerate(tqdm(reviews_data, desc="正在排版表格和处理图片"), start=2):
        ws.row_dimensions[row_idx].height = 190
        for col_idx, key in enumerate(fieldnames, start=1):
            val = review.get(key)
            ws.cell(row=row_idx, column=col_idx).value = str(val) if isinstance(val, (dict, list)) else val
        
        pictures = review.get('pictures')
        if isinstance(pictures, list):
            for pic_idx, img_url in enumerate(pictures):
                col_idx = len(fieldnames) + 1 + pic_idx
                if img_url.startswith('//'): img_url = 'https:' + img_url
                try:
                    img_res = requests.get(img_url, timeout=5)
                    if img_res.status_code == 200:
                        img_stream = io.BytesIO(img_res.content)
                        pil_img = PILImage.open(img_stream)
                        w, h = pil_img.size
                        ratio = min(120/w, 120/h) if w > 0 and h > 0 else 1
                        img_byte_arr = io.BytesIO()
                        pil_img.save(img_byte_arr, format='PNG')
                        img_byte_arr.seek(0)
                        xl_img = OpenpyxlImage(img_byte_arr)
                        xl_img.width = int(w * ratio)
                        xl_img.height = int(h * ratio)
                        xl_img.anchor = ws.cell(row=row_idx, column=col_idx).coordinate
                        ws.add_image(xl_img)
                except: pass
    
    wb.save(excel_path)
    print(f"\n🎉 成功！文件已保存在文件夹: {dir_name}")
    
    # 4. 打包并下载
    import shutil
    shutil.make_archive(dir_name, 'zip', dir_name)
    print(f"\n✅ 任务全部完成！正在打包并启动浏览器自动下载...")
    files.download(f"{dir_name}.zip")
else:
    print("未找到任何评价。")

🚀 任务开始...


正在抓取评论: 0条 [00:00, ?条/s]


🎨 正在生成包含图片的 Excel 表格...


正在排版表格和处理图片:   0%|          | 0/3 [00:00<?, ?it/s]


🎉 成功！文件已保存在文件夹: HADAY-Delicious-Light-Soy-Sauce_113281_20260403_170852_3_lite

✅ 任务全部完成！正在打包并启动浏览器自动下载...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>